# Lab 10: Convolutional Layers

**Computer Vision Course**

Building on Lab 9, today you'll see how the scale-space ideas you built by hand translate directly into **learnable convolutional layers** in a neural network.

**What you'll learn:**
- How convolutional layers generalise handcrafted filters
- The role of filter count, kernel size, and bias
- How ReLU activations change learning dynamics
- Why more parameters do not always mean better results

**What you'll build:**
- A baseline dense network (benchmark from Lab 6)
- A single convolutional layer network
- A two-layer convolutional network
- Variants exploring bias and activation

**Why this matters:**
In Lab 9 you hand-designed Gaussian filters and Laplacians. Today the network **learns those filters automatically** from data!

**Connection to previous labs:**
- Lab 7: Learned convolution and Gaussian smoothing
- Lab 8: Built edge detectors (Sobel, Laplacian, LoG)
- Lab 9: Applied edge detection across multiple scales
- Lab 10 (today): See how CNNs learn multi-scale features automatically!

## Setup

In [ ]:
"""
Computer Vision Course - Lab 10: Convolutional Layers

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 10: Convolutional Layers")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)

    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        !git -C computer-vision pull
        print("✓ Repository updated successfully")

    %cd computer-vision/labs/lab10_conv_layers
    print(f"✓ Current directory: {os.getcwd()}")

    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")

    print("-" * 60)
    print("🟢 Colab setup complete!\n")

else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")

    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")

    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Import course utilities
try:
    from sdx import plot_loss
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

print("✓ All imports successful")

## Loading the Training and Testing Data

We will use the [MNIST dataset](http://yann.lecun.com/exdb/mnist/) again, but probably for the last time.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"✓ Training samples : {len(train_dataset):,}")
print(f"✓ Testing samples  : {len(test_dataset):,}")

## Convenience Functions

Since we will use the same training parameters for all networks, we will define some functions to avoid repetition.

In [ ]:
def get_optimizer_and_loss(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters())
    return criterion, optimizer

print("✓ get_optimizer_and_loss defined")

In [ ]:
def fit_default_parameters(model, epochs=32):
    """
    Train model with default parameters and return loss history.
    """
    criterion, optimizer = get_optimizer_and_loss(model)
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # --- training ---
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        history['train_loss'].append(running_loss / len(train_loader.dataset))

        # --- validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in test_loader:
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
        history['val_loss'].append(val_loss / len(test_loader.dataset))

        print(f'Epoch {epoch+1:02d}/{epochs} — '
              f'train loss: {history["train_loss"][-1]:.4f}  '
              f'val loss: {history["val_loss"][-1]:.4f}')

    return history

print("✓ fit_default_parameters defined")

---
## Part 1: Revisiting the Single Dense Layer Network

Back in Lab 5, this network had the best overall results, since adding ReLU resulted in a smaller loss but also led to overfitting. So we will start with this network as benchmark.

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 397),
    nn.Linear(397, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

This is the smallest training loss we got, but we couldn't get rid of the oscillation in the testing loss.

### 💡 Key Insight

A dense layer treats every pixel **independently** — it has no notion of spatial structure.

**Convolutional layers** solve this by sharing weights across all positions in the image, exactly like the Gaussian and Laplacian filters you built in Labs 7–9!

---
## Part 2: Replacing the Dense Layer with a Convolutional Layer

Instead of the dense layer, we will now use a convolutional layer with three $3 \times 3$ filters.

The idea is inspired by the scale space, which is based on convolving the image with multiple Gaussian kernels.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3),   # (1, 28, 28) -> (3, 26, 26)
    nn.Flatten(),
    nn.Linear(3 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

The size of the output tensor is $3 \times 26 \times 26$, since the borders are ignored and there are three filters.

The number of parameters in the convolutional layer is $30$: for each of the three filters, there are $3 \times 3$ weights and one bias.

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
## Part 3: Adding Another Convolutional Layer

We will now add another convolutional layer that collapses the three channels. The idea is inspired by the scale space Laplacian.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3),   # (1, 28, 28) -> (3, 26, 26)
    nn.Conv2d(3, 1, kernel_size=1),   # (3, 26, 26) -> (1, 26, 26)
    nn.Flatten(),
    nn.Linear(1 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

The size of the output tensor is $1 \times 26 \times 26$, since the borders are not ignored and there is one filter.

The number of parameters in the second convolutional layer is $4$: there are $1 \times 1$ weights for each channel, plus one bias.

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
## Part 4: Removing Bias

To be more faithful to the idea of representing a scale space, we will add `bias=False` to disable the bias.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3, bias=False),   # (1, 28, 28) -> (3, 26, 26)
    nn.Conv2d(3, 1, kernel_size=1, bias=False),   # (3, 26, 26) -> (1, 26, 26)
    nn.Flatten(),
    nn.Linear(1 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

Notice the small difference in the number of parameters.

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
## Part 5: Re-Revisiting the Activation Parameter

Finally, we will add `nn.ReLU()` activation layers, which was what resulted in a smaller loss but also led to overfitting in Lab 5.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3, bias=False),   # (1, 28, 28) -> (3, 26, 26)
    nn.ReLU(),
    nn.Conv2d(3, 1, kernel_size=1, bias=False),   # (3, 26, 26) -> (1, 26, 26)
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(1 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

Notice that there is no difference in the number of parameters — `nn.ReLU()` has none.

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
## Part 6: Putting the Bias Back

Now that we have ReLU activations, let us put the bias back and observe whether it makes a difference.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3),   # (1, 28, 28) -> (3, 26, 26)
    nn.ReLU(),
    nn.Conv2d(3, 1, kernel_size=1),   # (3, 26, 26) -> (1, 26, 26)
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(1 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

Notice that we cannot simply add parameters and expect the results to always improve.

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
## 🧠 Connection to Scale Space

**Everything you hand-built in Lab 9 is what convolutional layers do automatically!**

### Scale Space → CNN Layers

**Lab 9 (handcrafted):**
1. ✅ Designed Gaussian filters manually with chosen σ values
2. ✅ Built Laplacian / DoG filters to detect edges
3. ✅ Decided how to consolidate across scales
4. ✅ Applied unsharp masking to enhance details

**Lab 10 (learned):**
1. ✅ `Conv2d(1, 3, kernel_size=3)` learns 3 filters — just like S_σ, S_2σ, S_3σ
2. ✅ `Conv2d(3, 1, kernel_size=1)` collapses channels — just like the DoG / Laplacian step
3. ✅ `nn.ReLU()` keeps only positive responses — like thresholding edge maps
4. ✅ The network decides **which filters matter** by minimising the loss

### The Key Difference

**Handcrafted filters (Labs 7–9):**
- You chose the filter shapes
- You chose σ values
- You decided the consolidation strategy

**Learned filters (Lab 10):**
- Network **learns optimal filters** from data
- Network **learns how to combine channels**
- Network **learns what features minimise the loss**

**But the underlying principles are exactly the same!**

---
## ✏️ Challenge

Can you squeeze more improvement from this point? This time, however, I do *not* want you to explore the API reference. This time, I want you to *explain* the modifications. Why do you think they make sense?

In [ ]:
# ── Your code here ──────────────────────────────────────────────────────────

model = nn.Sequential(
    nn.Conv2d(1, 3, kernel_size=3),
    nn.ReLU(),
    nn.Conv2d(3, 1, kernel_size=1),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(1 * 26 * 26, 10),
)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

# ────────────────────────────────────────────────────────────────────────────

print("Complete the challenge above")

*write your rationale here*

In [ ]:
history = fit_default_parameters(model)

plot_loss(history)

---
### ✏️ Final Reflection (+1 point)

Answer these questions:

1. **Why does a convolutional layer have far fewer parameters than a dense layer for the same image size?**

2. **What is the role of the $1 \times 1$ convolution in our two-layer network?**

3. **How does today's content connect to the scale-space pyramid from Lab 9?**

4. **Why might disabling bias be useful when trying to mimic a Gaussian filter?**

In [ ]:
# Your reflection:

# 1. Why fewer parameters than a dense layer?
#    ...

# 2. Role of the 1x1 convolution?
#    ...

# 3. Connection to Lab 9 scale-space pyramid?
#    ...

# 4. Why disable bias for Gaussian-like filters?
#    ...